In [1]:
import pandas as pd
import numpy as np

# ĐỌC VÀ KHÁM PHÁ DỮ LIỆU
print("--- ĐỌC VÀ KHÁM PHÁ DỮ LIỆU ---")
df = pd.read_csv('vgsales.csv')
print(f"Kích thước ban đầu: {df.shape}")
df.head()


--- ĐỌC VÀ KHÁM PHÁ DỮ LIỆU ---
Kích thước ban đầu: (16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [2]:
# 1. Kiểm tra khóa chính (Quang's strength)
print("\n--- KIỂM TRA DỮ LIỆU ---")
if 'Rank' in df.columns:
    duplicate_ranks = df.duplicated(subset=['Rank']).sum()
    print(f"Số dòng trùng Rank (Primary Key): {duplicate_ranks}")

print(f"Missing values ban đầu:\n{df.isnull().sum()[df.isnull().sum() > 0]}")



--- KIỂM TRA DỮ LIỆU ---
Số dòng trùng Rank (Primary Key): 0
Missing values ban đầu:
Year         271
Publisher     58
dtype: int64


In [3]:
# 2. Xử lý kiểu dữ liệu an toàn (Quang & Danh's strength)
print("\n--- XỬ LÝ KIỂU DỮ LIỆU & MISSING VALUES ---")
# Sử dụng errors='coerce' để test Year
year_test = pd.to_numeric(df['Year'], errors='coerce')
invalid_year_count = year_test.isnull().sum() - df['Year'].isnull().sum()
if invalid_year_count > 0:
    print(f"Phát hiện {invalid_year_count} giá trị Year bị lỗi định dạng.")

# Ép kiểu an toàn
df['Year'] = year_test

# Xóa missing Year (Khoi's logic for vgsales)
initial_len = len(df)
df = df.dropna(subset=['Year'])
print(f"Đã xóa {initial_len - len(df)} dòng bị thiếu năm phát hành.")
df['Year'] = df['Year'].astype(int)

# Điền Missing cho Publisher
df['Publisher'] = df['Publisher'].fillna('Unknown')



--- XỬ LÝ KIỂU DỮ LIỆU & MISSING VALUES ---
Đã xóa 271 dòng bị thiếu năm phát hành.


In [4]:
# 3. Chuẩn hóa chuỗi (Khoi's strength)
print("\n--- CHUẨN HÓA ĐỊNH DẠNG CHUỖI ---")
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()



--- CHUẨN HÓA ĐỊNH DẠNG CHUỖI ---


In [5]:
# 4. Xử lý trùng lặp chuyên sâu (Khoi's strength)
print("\n--- XỬ LÝ TRÙNG LẶP ---")
dups = df[df.duplicated(subset=['Name', 'Platform'], keep=False)]
print(f"Phát hiện {len(dups)} dòng trùng lặp (Cùng Game & Platform).")

agg_funcs = {
    'Year': 'first',
    'Genre': 'first',
    'Publisher': 'first',
    'NA_Sales': 'sum',
    'EU_Sales': 'sum',
    'JP_Sales': 'sum',
    'Other_Sales': 'sum',
    'Global_Sales': 'sum'
}
df = df.groupby(['Name', 'Platform'], as_index=False).agg(agg_funcs)
print("Đã gộp thành công các dòng trùng lặp bằng cách cộng dồn doanh thu.")



--- XỬ LÝ TRÙNG LẶP ---
Phát hiện 6 dòng trùng lặp (Cùng Game & Platform).
Đã gộp thành công các dòng trùng lặp bằng cách cộng dồn doanh thu.


In [6]:
# 5. Tính toán lại tổng doanh thu (Khoi's strength)
print("\n--- TÍNH TOÁN LẠI TỔNG DOANH THU ---")
df['Global_Sales'] = df['NA_Sales'] + df['EU_Sales'] + df['JP_Sales'] + df['Other_Sales']

# Sắp xếp và cấp lại Rank
df = df.sort_values('Global_Sales', ascending=False).reset_index(drop=True)
df['Rank'] = df.index + 1

# Căn chỉnh lại cột
cols = ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
df = df[cols]
df.head()



--- TÍNH TOÁN LẠI TỔNG DOANH THU ---


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.83
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.38


In [7]:
# 6. Kiểm tra lại và phân tích nhanh (Quang's strength)
print("\n--- KIỂM TRA LẠI DỮ LIỆU SAU KHI CLEAN ---")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Kích thước sau cùng: {df.shape}")

print("\n--- EDA CƠ BẢN ---")
print(f"Số lượng Genre: {df['Genre'].nunique()}")
print(f"Top 3 Platform:\n{df['Platform'].value_counts().head(3)}")

# Lưu file
output_file = 'vgsales-clean.csv'
df.to_csv(output_file, index=False)
print(f"\n✅ Đã lưu file sạch tại: {output_file}")



--- KIỂM TRA LẠI DỮ LIỆU SAU KHI CLEAN ---
Missing values: 0
Kích thước sau cùng: (16324, 11)

--- EDA CƠ BẢN ---
Số lượng Genre: 12
Top 3 Platform:
Platform
DS     2133
PS2    2127
PS3    1303
Name: count, dtype: int64



✅ Đã lưu file sạch tại: vgsales-clean.csv
